# v1.2 × M2 · 원본 가중치 CPU preflight / Tesla T4 비교
기본 실행은 모델 패키지 다운로드와 실제 가중치 CPU 합성 검사까지다. T4 비교에는 WORK.md의 고정 audit 및 RGB·pose 원본을 준비하고 아래 경로를 지정한다.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, tempfile, zipfile
from datetime import datetime, timezone
from google.colab import auth, drive
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import google.auth
import torch

RUN_T4 = False
BASELINE_ONLY = False
AUDIT_ZIP = ''  # original completed_003/reports.zip
RGB_ROOT = ''   # original extracted RGB tree
POSE_ROOT = ''  # original extracted pose tree

drive.mount('/content/drive')
auth.authenticate_user()
api=build('drive','v3',credentials=google.auth.default()[0])
work=Path(tempfile.mkdtemp(dir='/content'))
repo=work/'fork'
package=work/'package'
stage=work/'evidence'
stage.mkdir()
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
dest=Path('/content/drive/MyDrive/rewind_v12_m2')/stamp
dest.mkdir(parents=True,exist_ok=False)
revision='447385016e7973ace9871d53781c6f9153f83a57'
def digest(p):
    with p.open('rb') as f:
        return hashlib.file_digest(f,'sha256').hexdigest()
def run(*args,cwd=work,env=None,log=None):
    if log is None:
        subprocess.run(args,cwd=cwd,env=env,check=True)
    else:
        with (stage/log).open('w') as f:
            subprocess.run(args,cwd=cwd,env=env,check=True,stdout=f,stderr=subprocess.STDOUT)
def archive(label):
    path=Path(shutil.make_archive(str(work/label),'zip',stage))
    copy=dest/path.name
    shutil.copy2(path,copy)
    if digest(copy)!=digest(path):
        raise ValueError('archive readback failed')
    (dest/(label+'.sha256')).write_text(digest(copy))
    print(copy)
run('git','clone','-q','--branch','integration/v12-m2-refinement-20261122','https://github.com/darkkyoung/REWIND-like-prototype.git',str(repo))
run('git','checkout','--detach',revision,cwd=repo)

In [ ]:
fid='1eXSpkVdoLFybz6QBvX-uyrOFfZURfxdt'
meta=api.files().get(fileId=fid,fields='id,name,version,size,sha256Checksum',supportsAllDrives=True).execute()
if int(meta['size'])!=127886815:
    raise ValueError('unexpected v1.2 model package size')
zpath=work/'v12.zip'
with zpath.open('xb') as f:
    reader=MediaIoBaseDownload(f,api.files().get_media(fileId=fid,supportsAllDrives=True))
    done=False
    while not done:
        _,done=reader.next_chunk(num_retries=3)
if digest(zpath)!=meta['sha256Checksum']:
    raise ValueError('Drive archive identity mismatch')
with zipfile.ZipFile(zpath) as z:
    if z.testzip() is not None:
        raise ValueError('CRC')
    z.extractall(package)
source=package/'source'
env=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[source,repo])),CUBLAS_WORKSPACE_CONFIG=':4096:8')
(stage/'source.json').write_text(json.dumps(dict(fork=revision,package=meta,packaged_source=(source/'SOURCE_REVISION.txt').read_text().strip(),torch=str(torch.__version__)),indent=2))
try:
    run(sys.executable,'-m','unittest','-v','experiments.v12_m2.test_integration',cwd=source,env=env,log='cpu_tests.log')
    run(sys.executable,'-m','experiments.v12_m2.preflight',str(package),cwd=source,env=env,log='strict_cpu.log')
finally:
    archive('cpu_preflight')
print((stage/'strict_cpu.log').read_text())

In [ ]:
if RUN_T4:
    if not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name():
        raise RuntimeError('Tesla T4 required')
    if not all((AUDIT_ZIP,RGB_ROOT,POSE_ROOT)):
        raise ValueError('Set the three verified data paths in the first cell')
    args=[sys.executable,'-m','experiments.v12_m2.evaluate','--package',str(package),'--audit',AUDIT_ZIP,'--rgb',RGB_ROOT,'--pose',POSE_ROOT,'--device','cuda','--out',str(stage/'results')]
    if BASELINE_ONLY:
        args.append('--baseline-only')
    try:
        run(*args,cwd=source,env=env,log='evaluation.log')
    finally:
        archive('evaluation')
    print((stage/'results/metrics.csv').read_text())
else:
    print('CPU preflight only. Real validation inference has not run.')